# Elliptic-curve point addition

Append the six-stage reversible point addition and inspect its ports, source calls, and emitted primitive counts.

Adapted from the Qualtran v0.7.0 ECAdd tutorial. Copyright 2023 Google LLC. Licensed under Apache-2.0.

In [ ]:
from collections import Counter

from IPython.display import Markdown, SVG, display
from sympy import symbols

from examples.algorithms.field_addition_catalog import field_signature_svg
from flagquantum.algorithms.primitives.elliptic_curve_addition import (
    EllipticCurveAddition,
    elliptic_curve_addition_signature,
    elliptic_curve_addition_source_graph,
)
from flagquantum.algorithms.primitives.elliptic_curve_point import ECPoint
from flagquantum.drawer.call_graph import (
    export_call_graph_dot, format_call_graph_markdown, render_call_graph_svg,
)
from flagquantum.dynamic import DynamicCircuit

## Operation

`EllipticCurveAddition(width, modulus, window)` adds the point in `a,b` to the point in `x,y`, writing the sum into `x,y` on the source's clean domain. All five data words are big-endian Montgomery residues. The caller supplies the precomputed tangent in `lam_r`; `a,b,lam_r` pass through. A zero pair represents infinity. The window divides the width, with enough bits for `2p-1`.

The six stages follow [Litinski, figures 5 and 10](https://arxiv.org/abs/2306.08585). Reserve clean work for flags, slope, and shared arithmetic scratch. The locked circuit can leave a flag set or disagree with the mathematical sum on some inputs. Use the explicit [acceptance cases](../../docs/development/ECC_ADDITION_FILE_AUDIT.md) to distinguish the decomposed circuit from the point formula.

In [ ]:
n, p = symbols("n p")
symbolic_signature = elliptic_curve_addition_signature(n)
symbolic_signature

### Instances

The concrete declaration uses the source tutorial's width 5 and modulus 7. The symbolic declaration retains the same five through ports.

In [ ]:
example = EllipticCurveAddition(5, 7)
example.workspace_requirements()

In [ ]:
display(SVG(field_signature_svg((
    ("ECAdd(n,p)", symbolic_signature, {}),
    ("ECAdd(5,7)", example.source_signature, {}),
))))

### Calls

Each stage is called once. The graph and its leaf counts are source-level declarations.

In [ ]:
graph = elliptic_curve_addition_source_graph()
display(SVG(render_call_graph_svg(export_call_graph_dot(graph.root, graph.direct_calls))))
display(Markdown(format_call_graph_markdown(graph.root, graph.direct_calls, graph.leaf_counts)))
assert len(graph.leaf_counts) == 6 and set(graph.leaf_counts.values()) == {1}

### Bind the verified small circuit

The two input points are `(0,1)` on the curve `y²=x³+x+1` over the field of size 3. Their tangent is 2 and their sum is `(1,0)`. The initializer below uses the stored encoding. Read the bound circuit as IR to inspect its resource count; the channel tests independently verify the complex output and clean work.

In [ ]:
operation = EllipticCurveAddition(3, 3)
q, c = operation.workspace_requirements()
a, b, x, y, lam_r = (tuple(range(i * 3, (i + 1) * 3)) for i in range(5))
encoded = (0, 2, 0, 2, 1)
circuit = DynamicCircuit(15 + q)
for word, value in zip((a, b, x, y, lam_r), encoded, strict=True):
    for i, wire in enumerate(word):
        if value >> (2 - i) & 1:
            circuit.x(wire)
circuit.append(operation.make_on(
    a=a, b=b, x=x, y=y, lam_r=lam_r,
    workspace=tuple(range(15, 15 + q)), classical_bits=tuple(range(c)),
))
ir = circuit.to_ir()
actual = Counter(gate.name for gate in ir.instructions)
assert actual["measure"] == 1322 and actual["ccx"] == 19
point = ECPoint(0, 1, 3, 1)
summed = point + point
assert (summed.x, summed.y) == (1, 0)
{"bound_wires": ir.n_wires, "emitted": dict(actual), "expected_sum": (summed.x, summed.y)}